# Agent 第 02 课：工具调用协议总览（OpenAI function calling vs Hermes vs ReAct text）

第 01 课我们搭好了骨架，但模型还没有"手"——它只能输出文字，不能调工具。

这一课我们回答一个核心问题：

> **LLM 要怎么告诉外部代码"请帮我调用 XXX 工具，参数是 YYY"？**

答案不是一个，业界目前有**三种主流协议**，各有优劣。我们一个一个过，然后选一个作为本课程之后要一路用的协议。

提前说结论：**我们会选 Hermes 风格**。理由在第 4 节细讲。

## 1. 协议 A：OpenAI Function Calling（API 内建）

这是 OpenAI 2023 年 6 月推出的方案，现在几乎所有商业 API（OpenAI、Anthropic 的 tool use、Google Gemini）都支持。

**做法**：调用 API 时传一个 `tools` 参数，里面是 JSON Schema 描述；模型在输出时不是写文字，而是返回一个特殊字段 `tool_calls`。

```python
tools = [{
  'type': 'function',
  'function': {
    'name': 'calculator',
    'description': 'Evaluate a math expression',
    'parameters': {
      'type': 'object',
      'properties': {'expr': {'type': 'string'}},
      'required': ['expr']
    }
  }
}]
resp = client.chat.completions.create(model=..., tools=tools, messages=...)
# resp.choices[0].message.tool_calls  ← 模型选择调用的工具
```

**优点**：
- 干净，调用侧拿到的是结构化对象，不需要解析
- 现代商业模型（GPT-4o、Claude、Gemini）调用成功率很高

**缺点**：
- **黑箱**。模型是怎么决定调工具的？用了什么 prompt？API 层面看不到。换个 provider，行为可能微妙地变了你不知道
- 不同 provider 字段名不一样（OpenAI 的 `tool_calls` vs Anthropic 的 `tool_use` block），代码要兼容
- 本地开源模型支持参差——Qwen / Llama 新版本支持，但老模型、小模型要么没有，要么格式不稳定

## 2. 协议 B：ReAct Text Format（最朴素）

ReAct 原论文里用的，就是纯文本格式：

```
Thought: I should use the calculator.
Action: calculator
Action Input: 123 * 456
Observation: 56088
Thought: I now have the answer.
Final Answer: 56088
```

**优点**：
- **最透明**。你看到的就是模型看到的，没任何魔法
- 任何 LLM 都能用，哪怕是不支持 function calling 的老模型

**缺点**：
- 解析非常脆弱。模型偶尔会写成 `Action input:` 小写，或 `**Action**:` 加粗，你的正则就炸了
- 参数是自由文本，多参数、嵌套结构就很难表达
- 没有"这段已经结束"的明确分隔符，模型容易多嘴

## 3. 协议 C：Hermes Format（开源社区最香的折中）

NousResearch 训练的 **Hermes 系列模型**（Hermes 2 Pro、Hermes 3 等）推广了一种用 **XML 标签**把工具调用嵌在普通文本里的格式：

```
<thinking>
I should use the calculator for this multiplication.
</thinking>

<tool_call>
{"name": "calculator", "arguments": {"expr": "123 * 456"}}
</tool_call>
```

外部代码看到 `<tool_call>...</tool_call>` 标签就知道模型要调工具，解析里面的 JSON 就拿到 name + arguments。工具执行完之后，你把结果以类似的标签喂回去：

```
<tool_response>
{"name": "calculator", "content": 56088}
</tool_response>
```

模型接着生成下一步，要么继续调工具，要么出最终答案。

**为什么这是开源社区的最爱**：
1. **100% 在 prompt 里可见**——没有 API 层魔法，你换模型、换推理引擎都不用改协议
2. 参数用 JSON，多字段、嵌套都自然
3. `<tool_call>` 标签边界清晰，解析脆弱性比纯 ReAct 文本低得多
4. 很多开源模型（包括 Qwen 新版）在 post-training 里都见过这种格式，成功率已经可以
5. **可以 self-host**——本地 LM Studio、vLLM、Ollama 都能跑

## 4. 我们的选择：Hermes 风格

这门课走 **Hermes 风格**，原因：

- 本课程用本地 LLM，OpenAI function calling 在本地模型上表现不稳定
- Hermes 格式教学最清楚——**你能看见 agent 是怎么工作的**，哪一步模型想了什么、调了什么。没有任何黑箱
- 学会 Hermes 以后，迁移到 OpenAI / Anthropic / Bedrock 的内建 function calling 只是换个 adapter，核心循环代码不用变

**本课后续，你看到的所有 agent 都是这个格式。**

下面把它跑起来。

In [ ]:
from openai import OpenAI
client = OpenAI(base_url='http://10.0.0.63:1234/v1', api_key='lm-studio')
model_ids = [m.id for m in client.models.list().data]
chat_model = next((m for m in ['qwen/qwen3.6-35b-a3b','qwen3.6-35b-a3b','qwen3.5-35b-a3b'] if m in model_ids), model_ids[0])
print('chat =', chat_model)

## 5. 定义一个工具 + 写成 Hermes 系统 prompt

Hermes 风格下，"可用工具"不是 API 参数，而是**直接写在系统 prompt 里**。这是它的精髓——**对模型来说一切都在上下文里可见**。

In [ ]:
import json, re

# 工具的 JSON Schema 描述
TOOLS = [
    {
        'name': 'calculator',
        'description': 'Evaluate a Python-style math expression like "3*(4+5)".',
        'parameters': {
            'type': 'object',
            'properties': {'expr': {'type': 'string', 'description': 'The math expression'}},
            'required': ['expr']
        }
    }
]

HERMES_SYSTEM = f"""You are a helpful assistant that can call tools.

You have access to the following tools:
{json.dumps(TOOLS, indent=2)}

To call a tool, output EXACTLY this structure (and nothing else on that turn):

<tool_call>
{{"name": "<tool_name>", "arguments": {{ ... }}}}
</tool_call>

You will then receive a <tool_response>...</tool_response> block with the result.
When you have enough information to answer, respond in plain text starting with 'Final Answer:'.
Do not make up tool results. Do not call tools that are not listed above."""

print(HERMES_SYSTEM[:400], '...')

## 6. 解析器 + 工具执行器

两个小工具函数。这一课我们还不搭完整 agent 循环（那是第 03 课），这里只证明"解析 + 执行"的链路通了。

In [ ]:
TOOL_CALL_RE = re.compile(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', re.S)

def parse_tool_calls(text):
    """返回 [{'name':..., 'arguments':...}, ...]"""
    out = []
    for m in TOOL_CALL_RE.finditer(text):
        try:
            out.append(json.loads(m.group(1)))
        except json.JSONDecodeError:
            pass
    return out

def exec_tool(name, arguments):
    if name == 'calculator':
        # 教学用——生产里绝对不能直接 eval 任何字符串
        return {'result': eval(arguments['expr'], {'__builtins__': {}}, {})}
    return {'error': f'unknown tool: {name}'}

# 简单自检
sample = '<tool_call>\n{"name": "calculator", "arguments": {"expr": "2+2"}}\n</tool_call>'
calls = parse_tool_calls(sample)
print('parsed:', calls)
print('exec :', exec_tool(calls[0]['name'], calls[0]['arguments']))

## 7. 让模型真的输出 Hermes 格式看看

只做**一次**调用，观察模型的原始输出（还没写循环）。

In [ ]:
messages = [
    {'role': 'system', 'content': HERMES_SYSTEM},
    {'role': 'user', 'content': 'What is 1234 * 5678? Use the calculator.'}
]
r = client.chat.completions.create(model=chat_model, temperature=0, messages=messages)
raw = r.choices[0].message.content
print('--- raw LLM output ---')
print(raw)
print('--- parsed ---')
print(parse_tool_calls(raw))

如果本地模型吃下了 Hermes 格式，你会看到 `<tool_call>` 标签和 JSON。如果它偶尔漏标签、或写成别的变体，别慌——第 03 课我们会加容错。

## 8. 工程直觉

1. **三种协议本质一样**：都是"模型输出→代码理解"。区别只是这个接口放在 API 层还是 prompt 层。
2. **Hermes 的精神是"一切在 prompt 里"**。这让 agent 变得可调试：卡住了，打印 messages，一眼看出哪一步错了。API 内建的 function calling 做不到这点。
3. **工具 schema 要写清楚 description**。模型选工具 80% 依赖这个字段。写 "Evaluate math" vs "Evaluate a Python-style math expression like 3*(4+5)"，成功率差很大。
4. **eval 只是教学占位**。真实工具一定要有参数校验（pydantic）、超时、沙箱。第 04 课会改。
5. **本地模型的 Hermes 格式支持不是 100% 稳的**。Qwen 3.6 还不错，小模型可能会漏标签。所以我们解析器要写得宽容。

---

下一课：**第 03 课 手写真正的 ReAct Agent**——把今天的解析 + 执行接进一个带循环的 agent，让它真正多步骤地完成任务。这是全课程最重要的一课。